# # PRE_12 — Cadena de suministro (envíos de salud pública)
#
# **Qué se hace:** se limpia `data/supply_chain.csv` (10.324 líneas de pedido, 33 columnas) y se generan
# 8 tablas en `submission/`: KPIs, resúmenes por país, modo de envío, país x modo y mes, costos de
# flete por modo, y listas de países y segmentos prioritarios.
#
# **Decisiones de limpieza** (se justifican en la sección 1):
# - Fechas: `Scheduled Delivery Date`, `Delivered to Client Date` y `Delivery Recorded Date` vienen como `d-Mon-yy`;
#   las dos primeras columnas de fecha mezclan `m/d/yy` con textos ("Pre-PQ Process", "N/A - From RDC", "Date Not Captured") y
#   no se usan en los cálculos.
# - `Freight Cost (USD)` y `Weight (Kilograms)` traen texto en vez de número ("Freight Included in Commodity Cost",
#   "Invoiced Separately", "Weight Captured Separately", "See DN-xxx (ID#:...)"). Se convierten a número, y el texto se conserva
#   como estado (`freight_status`). **No se imputa**: un flete "incluido en el costo" o "ver otra línea" no es cero.
# - `Shipment Mode` vacío -> "Sin dato". Las sumas de flete solo usan líneas con flete numérico.
#
# **Definiciones**
# - *Entrega tardía* = `Delivered to Client Date` posterior a `Scheduled Delivery Date`. *Días de atraso* = entregada - programada.
# - *Flete % del valor* = flete conocido / valor de línea de esas mismas líneas (para no mezclar líneas sin flete).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Funciona tanto desde la carpeta de la actividad como desde notebooks/ (Jupyter)
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
SUBMISSION = ROOT / "submission"
SUBMISSION.mkdir(exist_ok=True)

raw = pd.read_csv(ROOT / "data" / "supply_chain.csv", low_memory=False)
print(raw.shape)
raw.iloc[:3, :12]

(10324, 33)


,ID,Project Code,PQ #,PO / SO #,ASN/DN #,Country,Managed By,Fulfill Via,Vendor INCO Term,Shipment Mode,PQ First Sent to Client Date,PO Sent to Vendor Date
0,1,100-CI-T01,Pre-PQ Process,SCMS-4,ASN-8,Côte d'Ivoire,PMO - US,Direct Drop,EXW,Air,Pre-PQ Process,Date Not Captured
1,3,108-VN-T01,Pre-PQ Process,SCMS-13,ASN-85,Vietnam,PMO - US,Direct Drop,EXW,Air,Pre-PQ Process,Date Not Captured
2,4,100-CI-T01,Pre-PQ Process,SCMS-20,ASN-14,Côte d'Ivoire,PMO - US,Direct Drop,FCA,Air,Pre-PQ Process,Date Not Captured


# ## 1. Calidad de datos y limpieza

In [2]:
print("Filas duplicadas:", int(raw.duplicated().sum()), "| ID repetidos:", int(raw.ID.duplicated().sum()))
print("Nulos:\n", raw.isna().sum()[lambda s: s > 0].to_string())
for c in ["Freight Cost (USD)", "Weight (Kilograms)"]:
    num = pd.to_numeric(raw[c], errors="coerce")
    print(f"\n{c}: {int(num.isna().sum())} no numéricos ({num.isna().mean():.1%}). Más frecuentes:")
    print(raw.loc[num.isna(), c].str.replace(r"See .*", "See <otra línea>", regex=True).value_counts().head(5).to_string())

Filas duplicadas: 0 | ID repetidos: 0
Nulos:
 Shipment Mode                 360
Dosage                       1736
Line Item Insurance (USD)     287

Freight Cost (USD): 4126 no numéricos (40.0%). Más frecuentes:
Freight Cost (USD)
See <otra línea>                      2445
Freight Included in Commodity Cost    1442
Invoiced Separately                    239

Weight (Kilograms): 3952 no numéricos (38.3%). Más frecuentes:
Weight (Kilograms)
See <otra línea>              2445
Weight Captured Separately    1507


In [3]:
df = raw.copy()
df["freight_usd"] = pd.to_numeric(df["Freight Cost (USD)"], errors="coerce")
df["weight_kg"] = pd.to_numeric(df["Weight (Kilograms)"], errors="coerce")


def estado_flete(txt):
    """Clasifica el contenido original de Freight Cost (USD)."""
    if pd.notna(pd.to_numeric(txt, errors="coerce")):
        return "Con flete"
    if txt == "Freight Included in Commodity Cost":
        return "Incluido en costo"
    if txt == "Invoiced Separately":
        return "Facturado aparte"
    if str(txt).startswith("See "):
        return "Ver otra línea"
    return "Otro"


df["freight_status"] = df["Freight Cost (USD)"].map(estado_flete)
print(df.freight_status.value_counts().to_string())
assert (df.freight_status != "Otro").all()
assert (df.freight_status == "Con flete").sum() == df.freight_usd.notna().sum()

for c, n in [("Scheduled Delivery Date", "scheduled"), ("Delivered to Client Date", "delivered"), ("Delivery Recorded Date", "recorded")]:
    df[n] = pd.to_datetime(df[c], format="%d-%b-%y", errors="coerce")
    print(c, "no interpretables:", int(df[n].isna().sum()), "| rango:", df[n].min().date(), "->", df[n].max().date())

df["Shipment Mode"] = df["Shipment Mode"].fillna("Sin dato")
df["country"], df["mode"] = df["Country"], df["Shipment Mode"]
df["value"] = df["Line Item Value"]
df["delay_days"] = (df.delivered - df.scheduled).dt.days
df["is_late"] = df.delay_days > 0
df["month"] = df.delivered.dt.strftime("%Y-%m")
df["freight_value"] = df.value.where(df.freight_usd.notna())  # valor de las líneas que sí tienen flete

print("\nValores negativos (valor/flete/cantidad):", int((df.value < 0).sum()), int((df.freight_usd < 0).sum()), int((df["Line Item Quantity"] < 0).sum()))
print("Líneas con valor 0:", int((df.value == 0).sum()), "| países:", df.country.nunique(), "| modos:", df["mode"].unique().tolist())
print("Atraso (días): min", df.delay_days.min(), "máx", df.delay_days.max(), "| tardías:", f"{df.is_late.mean():.1%}")

freight_status
Con flete            6198
Ver otra línea       2445
Incluido en costo    1442
Facturado aparte      239
Scheduled Delivery Date no interpretables: 0 | rango: 2006-05-02 -> 2015-12-31
Delivered to Client Date no interpretables: 0 | rango: 2006-05-02 -> 2015-09-14
Delivery Recorded Date no interpretables: 0 | rango: 2006-05-02 -> 2015-09-14

Valores negativos (valor/flete/cantidad): 0 0 0
Líneas con valor 0: 17 | países: 43 | modos: ['Air', 'Sin dato', 'Truck', 'Air Charter', 'Ocean']
Atraso (días): min -372 máx 192 | tardías: 11.5%


# **Conclusión:** sin duplicados ni IDs repetidos. 40 % de las líneas no tiene un flete numérico (24 % remite a otra línea del
# mismo envío, 14 % va incluido en el costo del producto y 2 % se factura aparte): ese vacío es información, no un error, y por eso no se rellena con 0 ni
# con la media. Las 3 fechas de entrega se interpretan completas; hay entregas adelantadas (hasta 372 días antes de lo programado), por eso el
# atraso medio global es negativo (-6 días) y se reporta aparte el atraso de las tardías. 17 líneas tienen valor 0 y se conservan. `Shipment Mode` tiene 360 vacíos que se agrupan como "Sin dato".

In [4]:
def resumen(d, by):
    g = d.groupby(by).agg(
        lines=("ID", "count"),
        quantity=("Line Item Quantity", "sum"),
        value=("value", "sum"),
        lines_with_freight=("freight_usd", "count"),
        freight_usd=("freight_usd", "sum"),
        freight_base_value=("freight_value", "sum"),
        late_lines=("is_late", "sum"),
        avg_delay_days=("delay_days", "mean"),
    )
    g["freight_pct_of_value"] = (g.freight_usd / g.freight_base_value.replace(0, np.nan)).round(4)
    g["freight_coverage"] = (g.lines_with_freight / g.lines).round(4)
    g["late_rate"] = (g.late_lines / g.lines).round(4)
    g["value_share"] = (g.value / d.value.sum()).round(4)
    g[["value", "freight_usd", "freight_base_value", "avg_delay_days"]] = g[["value", "freight_usd", "freight_base_value", "avg_delay_days"]].round(2)
    return g.drop(columns="freight_base_value").reset_index()


total_value, total_freight = df.value.sum(), df.freight_usd.sum()
late_rate_all = df.is_late.mean()

# ## 2. KPIs globales (`overall_kpis.csv`)

In [5]:
known = df.freight_usd.notna()
overall = pd.DataFrame([
    ("order_lines", len(df)),
    ("countries", df.country.nunique()),
    ("vendors", df.Vendor.nunique()),
    ("total_quantity", int(df["Line Item Quantity"].sum())),
    ("total_line_value_usd", round(total_value, 2)),
    ("lines_with_numeric_freight", int(known.sum())),
    ("freight_coverage", round(known.mean(), 4)),
    ("total_freight_usd_known", round(total_freight, 2)),
    ("freight_pct_of_value_known_lines", round(total_freight / df.freight_value.sum(), 4)),
    ("late_lines", int(df.is_late.sum())),
    ("late_rate", round(late_rate_all, 4)),
    ("avg_delay_days", round(df.delay_days.mean(), 2)),
    ("avg_delay_days_when_late", round(df.delay_days[df.is_late].mean(), 2)),
    ("first_delivery", df.delivered.min().date().isoformat()),
    ("last_delivery", df.delivered.max().date().isoformat()),
], columns=["metric", "value"])
overall.to_csv(SUBMISSION / "overall_kpis.csv", index=False)
overall

,metric,value
0,order_lines,10324
1,countries,43
2,vendors,73
3,total_quantity,189265090
4,total_line_value_usd,1627584457.29
5,lines_with_numeric_freight,6198
6,freight_coverage,0.6003
7,total_freight_usd_known,68817849.41
8,freight_pct_of_value_known_lines,0.0579
9,late_lines,1186


# ## 3. Modo de envío y flete
# `mode_summary.csv`: indicadores por modo. `freight_by_mode.csv`: para cada modo, cómo se reparten las líneas por estado de
# flete y el costo de flete (total, medio, mediano, por kg cuando hay peso numérico).

In [6]:
mode = resumen(df, "mode").sort_values("value", ascending=False)
mode.to_csv(SUBMISSION / "mode_summary.csv", index=False)

fm = df.groupby(["mode", "freight_status"]).agg(
    lines=("ID", "count"),
    freight_usd=("freight_usd", "sum"),
    avg_freight_usd=("freight_usd", "mean"),
    median_freight_usd=("freight_usd", "median"),
).reset_index()
fm["share_of_mode_lines"] = (fm.lines / fm.groupby("mode").lines.transform("sum")).round(4)
both = df[df.freight_usd.notna() & df.weight_kg.notna() & (df.weight_kg > 0)]
per_kg = (both.groupby("mode").freight_usd.sum() / both.groupby("mode").weight_kg.sum()).round(3)
fm["freight_usd_per_kg"] = fm["mode"].map(per_kg).where(fm.freight_status == "Con flete")
fm[["freight_usd", "avg_freight_usd", "median_freight_usd"]] = fm[["freight_usd", "avg_freight_usd", "median_freight_usd"]].round(2)
fm = fm.sort_values(["mode", "lines"], ascending=[True, False])
fm.to_csv(SUBMISSION / "freight_by_mode.csv", index=False)
display(mode, fm)

,mode,lines,quantity,value,lines_with_freight,freight_usd,late_lines,avg_delay_days,freight_pct_of_value,freight_coverage,late_rate,value_share
0,Air,6113,55895073,6.272859e+08,4115,43038623.50,587,-3.76,0.0848,0.6732,0.0960,0.3854
4,Truck,2830,75852947,5.873824e+08,1166,11865688.23,455,-9.92,0.0322,0.4120,0.1608,0.3609
1,Air Charter,650,29479420,2.463722e+08,424,8926108.48,75,-19.04,0.0500,0.6523,0.1154,0.1514
2,Ocean,371,24231751,1.261779e+08,282,3590728.79,65,5.87,0.0340,0.7601,0.1752,0.0775
3,Sin dato,360,3805899,4.036598e+07,211,1396700.41,4,-2.51,0.0485,0.5861,0.0111,0.0248


,mode,freight_status,lines,freight_usd,avg_freight_usd,median_freight_usd,share_of_mode_lines,freight_usd_per_kg
0,Air,Con flete,4115,43038623.50,10458.96,4764.16,0.6732,5.263
3,Air,Ver otra línea,1474,0.00,NaN,NaN,0.2411,NaN
2,Air,Incluido en costo,334,0.00,NaN,NaN,0.0546,NaN
1,Air,Facturado aparte,190,0.00,NaN,NaN,0.0311,NaN
4,Air Charter,Con flete,424,8926108.48,21052.14,15322.73,0.6523,2.733
7,Air Charter,Ver otra línea,153,0.00,NaN,NaN,0.2354,NaN
6,Air Charter,Incluido en costo,71,0.00,NaN,NaN,0.1092,NaN
5,Air Charter,Facturado aparte,2,0.00,NaN,NaN,0.0031,NaN
8,Ocean,Con flete,282,3590728.79,12733.08,9520.98,0.7601,1.851
11,Ocean,Ver otra línea,44,0.00,NaN,NaN,0.1186,NaN


# ## 4. País, país x modo y mes

In [7]:
country = resumen(df, "country").sort_values("value", ascending=False)
country.to_csv(SUBMISSION / "country_summary.csv", index=False)

cm = resumen(df, ["country", "mode"]).sort_values(["country", "value"], ascending=[True, False])
cm.to_csv(SUBMISSION / "country_mode_summary.csv", index=False)

monthly = resumen(df, "month").sort_values("month")
monthly["value_mom_change"] = monthly.value.pct_change().round(4)
monthly.to_csv(SUBMISSION / "monthly_summary.csv", index=False)
display(country.head(10), monthly.tail(5))

,country,lines,quantity,value,lines_with_freight,freight_usd,late_lines,avg_delay_days,freight_pct_of_value,freight_coverage,late_rate,value_share
28,Nigeria,1194,33967950,3.502721e+08,764,14268550.39,142,-11.22,0.0544,0.6399,0.1189,0.2152
41,Zambia,683,28627539,2.469888e+08,517,6310437.21,108,-5.23,0.0347,0.7570,0.1581,0.1518
26,Mozambique,631,19252134,1.820247e+08,365,3294121.06,116,-1.21,0.0292,0.5784,0.1838,0.1118
37,Tanzania,519,12482994,1.294595e+08,412,4827989.32,66,-4.39,0.0470,0.7938,0.1272,0.0795
9,Côte d'Ivoire,1083,11898572,1.208845e+08,729,6092165.51,134,-5.38,0.0757,0.6731,0.1237,0.0743
33,South Africa,1406,22995781,1.086701e+08,289,2493587.30,115,-12.22,0.0284,0.2055,0.0818,0.0668
42,Zimbabwe,538,17407714,1.044400e+08,319,4318000.86,77,-11.01,0.0662,0.5929,0.1431,0.0642
39,Uganda,779,11914117,9.619734e+07,564,5484292.24,98,-7.58,0.0762,0.7240,0.1258,0.0591
30,Rwanda,430,8855350,7.112758e+07,350,5893493.23,37,-5.29,0.1033,0.8140,0.0860,0.0437
40,Vietnam,688,7104544,5.697441e+07,446,2737277.29,6,-0.35,0.0634,0.6483,0.0087,0.0350


,month,lines,quantity,value,lines_with_freight,freight_usd,late_lines,avg_delay_days,freight_pct_of_value,freight_coverage,late_rate,value_share,value_mom_change
108,2015-05,137,3347388,33091342.94,56,990528.18,6,-6.70,0.0461,0.4088,0.0438,0.0203,0.2528
109,2015-06,100,2866818,24973255.97,43,705313.86,10,-3.66,0.0474,0.4300,0.1000,0.0153,-0.2453
110,2015-07,170,1302480,12289553.96,32,203976.67,20,0.89,0.0804,0.1882,0.1176,0.0076,-0.5079
111,2015-08,121,2263591,18539307.35,6,80940.09,19,-3.42,0.1640,0.0496,0.1570,0.0114,0.5085
112,2015-09,21,171323,1662612.00,0,0.00,0,0.00,NaN,0.0000,0.0000,0.0010,-0.9103


# ## 5. Prioridades
# La prioridad mezcla **tamaño** y **desempeño**: se mira el valor de lo entregado tarde (`late_value`).
# - Se evalúan países (o segmentos) con al menos 50 líneas, para no sacar conclusiones de muestras muy pequeñas.
# - Prioridad **alta**: tasa de entrega tardía mayor que la global y valor por encima de la mediana del grupo evaluado.
# - **media**: cumple solo una de las dos; **baja**: ninguna.
# - `late_value_share` = parte del valor tardío total que está en esa fila; las listas se ordenan por `late_value`.

In [8]:
def prioridad(d, by, min_lines=50):
    g = d.assign(late_value=d.value.where(d.is_late, 0)).groupby(by).agg(
        lines=("ID", "count"), value=("value", "sum"), late_lines=("is_late", "sum"),
        late_value=("late_value", "sum"), avg_delay_days=("delay_days", "mean"),
        freight_usd=("freight_usd", "sum"), lines_with_freight=("freight_usd", "count"),
    )
    g["late_rate"] = g.late_lines / g.lines
    g["late_value_share"] = g.late_value / d.value.where(d.is_late, 0).sum()
    g["freight_coverage"] = g.lines_with_freight / g.lines
    g = g[g.lines >= min_lines]
    big, slow = g.value >= g.value.median(), g.late_rate > late_rate_all
    g["priority"] = np.select([big & slow, big | slow], ["alta", "media"], default="baja")
    g = g.round({"value": 2, "late_value": 2, "avg_delay_days": 2, "freight_usd": 2, "late_rate": 4,
                 "late_value_share": 4, "freight_coverage": 4})
    return g.drop(columns="lines_with_freight").sort_values("late_value", ascending=False).reset_index()


pc = prioridad(df, "country")
pc.insert(0, "rank", range(1, len(pc) + 1))
pc.to_csv(SUBMISSION / "priority_countries.csv", index=False)

ps = prioridad(df, ["mode", "Product Group"])
ps.insert(0, "rank", range(1, len(ps) + 1))
ps.to_csv(SUBMISSION / "priority_segments.csv", index=False)
display(pc.head(10), ps.head(10))

,rank,country,lines,value,late_lines,late_value,avg_delay_days,freight_usd,late_rate,late_value_share,freight_coverage,priority
0,1,Mozambique,631,1.820247e+08,116,53259611.47,-1.21,3294121.06,0.1838,0.2056,0.5784,alta
1,2,Nigeria,1194,3.502721e+08,142,49758109.31,-11.22,14268550.39,0.1189,0.1921,0.6399,alta
2,3,Zambia,683,2.469888e+08,108,33453393.04,-5.23,6310437.21,0.1581,0.1291,0.7570,alta
3,4,South Africa,1406,1.086701e+08,115,24787632.40,-12.22,2493587.30,0.0818,0.0957,0.2055,media
4,5,Tanzania,519,1.294595e+08,66,19606062.44,-4.39,4827989.32,0.1272,0.0757,0.7938,alta
5,6,Côte d'Ivoire,1083,1.208845e+08,134,17476127.74,-5.38,6092165.51,0.1237,0.0675,0.6731,alta
6,7,Uganda,779,9.619734e+07,98,16871823.08,-7.58,5484292.24,0.1258,0.0651,0.7240,alta
7,8,Zimbabwe,538,1.044400e+08,77,16211086.21,-11.01,4318000.86,0.1431,0.0626,0.5929,alta
8,9,Rwanda,430,7.112758e+07,37,7206078.09,-5.29,5893493.23,0.0860,0.0278,0.8140,media
9,10,Kenya,111,3.393156e+07,14,5669739.00,0.89,1429011.64,0.1261,0.0219,0.8198,media


,rank,mode,Product Group,lines,value,late_lines,late_value,avg_delay_days,freight_usd,late_rate,late_value_share,freight_coverage,priority
0,1,Truck,ARV,2780,5.823436e+08,454,1.426773e+08,-10.09,11832816.11,0.1633,0.5508,0.4115,alta
1,2,Air,ARV,4444,4.215357e+08,475,4.001365e+07,-5.17,25408562.95,0.1069,0.1545,0.6229,media
2,3,Air Charter,ARV,650,2.463722e+08,75,3.099997e+07,-19.04,8926108.48,0.1154,0.1197,0.6523,alta
3,4,Air,HRDT,1624,2.047452e+08,112,2.772041e+07,0.13,17357382.76,0.0690,0.1070,0.8067,media
4,5,Ocean,ARV,367,1.248509e+08,65,1.676712e+07,5.93,3518341.63,0.1771,0.0647,0.7602,media
5,6,Sin dato,ARV,309,3.811764e+07,4,6.949028e+05,-2.30,1267855.55,0.0129,0.0027,0.5663,baja
6,7,Truck,HRDT,50,5.038805e+06,1,1.609375e+05,-0.84,32872.12,0.0200,0.0006,0.4400,baja
7,8,Sin dato,HRDT,50,2.233708e+06,0,0.000000e+00,-3.84,128658.61,0.0000,0.0000,0.7000,baja


# ## 6. Verificación de consistencia

In [9]:
tol = 0.05
for name, t in [("mode", mode), ("country", country), ("country x mode", cm), ("mes", monthly)]:
    assert t.lines.sum() == len(df), name
    assert abs(t.value.sum() - total_value) < tol, name
    assert abs(t.freight_usd.sum() - total_freight) < tol, name
assert abs(fm.freight_usd.sum() - total_freight) < tol and fm.lines.sum() == len(df)
assert monthly.late_lines.sum() == df.is_late.sum()
assert pc.late_lines.sum() <= df.is_late.sum()
for name in ["country_mode_summary", "country_summary", "freight_by_mode", "mode_summary", "monthly_summary",
             "overall_kpis", "priority_countries", "priority_segments"]:
    print(f"{name}.csv", len(pd.read_csv(SUBMISSION / f"{name}.csv")), "filas")

country_mode_summary.csv 97 filas
country_summary.csv 43 filas
freight_by_mode.csv 19 filas
mode_summary.csv 5 filas
monthly_summary.csv 113 filas
overall_kpis.csv 15 filas
priority_countries.csv 22 filas
priority_segments.csv 8 filas


# ## Hallazgos

In [10]:
print(f"Valor total {total_value:,.0f} USD | flete conocido {total_freight:,.0f} USD ({total_freight / df.freight_value.sum():.1%} del valor de esas líneas)")
print(f"Entregas tardías: {late_rate_all:.1%} | atraso medio cuando es tardía: {df.delay_days[df.is_late].mean():.0f} días")
print(mode[["mode", "lines", "value_share", "late_rate", "freight_pct_of_value", "freight_coverage"]].to_string(index=False))
print("\nTop 5 países por valor:", country.head(5)[["country", "value_share"]].values.tolist())
print("Prioridad alta (países):", pc[pc.priority == "alta"].country.tolist())
print("Prioridad alta (segmentos):", ps[ps.priority == "alta"][["mode", "Product Group"]].values.tolist())

Valor total 1,627,584,457 USD | flete conocido 68,817,849 USD (5.8% del valor de esas líneas)
Entregas tardías: 11.5% | atraso medio cuando es tardía: 21 días
       mode  lines  value_share  late_rate  freight_pct_of_value  freight_coverage
        Air   6113       0.3854     0.0960                0.0848            0.6732
      Truck   2830       0.3609     0.1608                0.0322            0.4120
Air Charter    650       0.1514     0.1154                0.0500            0.6523
      Ocean    371       0.0775     0.1752                0.0340            0.7601
   Sin dato    360       0.0248     0.0111                0.0485            0.5861

Top 5 países por valor: [['Nigeria', 0.2152], ['Zambia', 0.1518], ['Mozambique', 0.1118], ['Tanzania', 0.0795], ["Côte d'Ivoire", 0.0743]]
Prioridad alta (países): ['Mozambique', 'Nigeria', 'Zambia', 'Tanzania', "Côte d'Ivoire", 'Uganda', 'Zimbabwe']
Prioridad alta (segmentos): [['Truck', 'ARV'], ['Air Charter', 'ARV']]


# - El flete tiene un vacío estructural grande: 4 de cada 10 líneas no trae un monto numérico, así que cualquier total de
#   flete es un **piso** (lo conocido), no el gasto real. Las cifras por país y modo incluyen `freight_coverage` para leerlas con ese contexto.
# - El modo aéreo concentra la mayoría de las líneas (59 %) y es el más caro: el flete conocido es ~8,5 % del valor frente a ~3,2 % en camión y ~3,4 % en
#   marítimo. Camión y marítimo son más tardíos (16 % y 18 % vs 10 % aéreo). Detalle en `freight_by_mode.csv`.
# - Nigeria (22 %), Zambia (15 %) y Mozambique (11 %) concentran casi la mitad del valor; 11,5 % de las entregas llega tarde (~21 días de atraso medio).
# - Los países y segmentos de prioridad alta son los que combinan mucho valor con más entregas tardías que el promedio global:
#   ahí un mismo punto de mejora en puntualidad protege más valor.